# Long Context Inference (32K+ Tokens)

**Stage 2: Memory Optimization - Notebook 23**

This notebook explores techniques for handling very long contexts (32K+ tokens) in LLM inference. We'll cover:
- Challenges with 32K+ token sequences
- Memory requirements and bottlenecks
- RoPE (Rotary Position Embedding) scaling
- YaRN (Yet another RoPE extensioN)
- Linear RoPE scaling vs NTK-aware scaling
- Memory management strategies
- Attention pattern visualization
- Combining Flash Attention + GQA + RoPE scaling
- Practical examples with long documents

**References**:
- LLM_INFERENCE_ROADMAP.md - Stage 2
- Notebook 20: Flash Attention Explained
- Notebook 21: MQA/GQA Implementation
- Notebook 55: KV Cache Optimization
- Paper: "Extending Context Window of Large Language Models via Position Interpolation" (Chen et al., 2023)
- Paper: "YaRN: Efficient Context Window Extension" (Peng et al., 2023)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Optional, Tuple
import math

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. The Long Context Challenge

### Why Long Contexts Are Hard

**Memory Requirements Scale with Context Length**:
```
Components that scale with sequence length:
1. KV Cache: O(n) - Linear scaling
2. Attention Matrix (standard): O(n²) - Quadratic!
3. Position Embeddings: O(n) - Linear

Example (Llama 2 70B):
- 4K context:  ~11 GB KV cache
- 8K context:  ~22 GB KV cache  
- 16K context: ~44 GB KV cache
- 32K context: ~88 GB KV cache (exceeds single A100!)
```

**Historical Context Limits**:
```
2020: GPT-3: 2048 tokens
2021: GPT-3.5: 4096 tokens
2022: Claude 1: 9K tokens
2023: GPT-4: 8K/32K tokens
2023: Claude 2: 100K tokens
2024: Claude 3: 200K tokens, Gemini 1.5: 1M tokens!
```

**Why We Need Long Contexts**:
- Document analysis (research papers, legal documents)
- Code repositories (entire codebases)
- Multi-turn conversations (chat history)
- Book summarization
- Video transcripts

In [ ]:
def calculate_long_context_memory(seq_len, model_config, batch_size=1, dtype_bytes=2):
    """
    Calculate total memory requirements for long context inference.
    
    Args:
        seq_len: Sequence length
        model_config: Dict with model parameters
        batch_size: Batch size
        dtype_bytes: Bytes per element (2 for FP16)
    """
    num_heads = model_config['num_heads']
    num_kv_heads = model_config.get('num_kv_heads', num_heads)
    head_dim = model_config['head_dim']
    num_layers = model_config['num_layers']
    d_model = model_config['d_model']
    
    # KV cache
    kv_cache_bytes = 2 * batch_size * seq_len * num_kv_heads * head_dim * num_layers * dtype_bytes
    
    # Attention matrix (if not using Flash Attention)
    attn_matrix_bytes = batch_size * num_heads * seq_len * seq_len * num_layers * dtype_bytes
    
    # Activations (rough estimate)
    activation_bytes = batch_size * seq_len * d_model * num_layers * dtype_bytes * 4  # 4 activations per layer
    
    return {
        'kv_cache_gb': kv_cache_bytes / 1e9,
        'attn_matrix_gb': attn_matrix_bytes / 1e9,
        'activation_gb': activation_bytes / 1e9,
        'total_no_flash_gb': (kv_cache_bytes + attn_matrix_bytes + activation_bytes) / 1e9,
        'total_with_flash_gb': (kv_cache_bytes + activation_bytes) / 1e9,  # No attn matrix
    }


# Llama 2 70B configuration
llama_70b_config = {
    'num_heads': 64,
    'num_kv_heads': 8,  # GQA
    'head_dim': 128,
    'num_layers': 80,
    'd_model': 8192,
}

print("Long Context Memory Requirements (Llama 2 70B)")
print("=" * 100)

context_lengths = [2048, 4096, 8192, 16384, 32768, 65536, 131072]

print(f"{'Context':<12} {'KV Cache':<12} {'Attn Matrix':<15} {'Total (Std)':<15} {'Total (Flash)':<15} {'Feasible?'}")
print("-" * 100)

results = []
for seq_len in context_lengths:
    mem = calculate_long_context_memory(seq_len, llama_70b_config)
    results.append(mem)
    
    # Check if feasible on A100 80GB
    model_size = 140  # GB for 70B model in FP16
    total_with_model = model_size + mem['total_with_flash_gb']
    feasible = "✓" if total_with_model < 80 else "✗ OOM"
    
    print(f"{seq_len:<12} {mem['kv_cache_gb']:<12.1f} {mem['attn_matrix_gb']:<15.1f} "
          f"{mem['total_no_flash_gb']:<15.1f} {mem['total_with_flash_gb']:<15.1f} {feasible}")

print("\n" + "=" * 100)
print("\nKey Observations:")
print("1. Without Flash Attention: OOM at 8K context")
print("2. With Flash Attention + GQA: Can handle up to 16K on single A100")
print("3. Beyond 16K: Need additional optimizations (quantization, multi-GPU)")
print("4. KV cache dominates memory usage for long contexts")

In [ ]:
# Visualize memory scaling
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Memory components
ax = axes[0]
kv_cache = [r['kv_cache_gb'] for r in results]
attn_matrix = [r['attn_matrix_gb'] for r in results]
activation = [r['activation_gb'] for r in results]

ax.plot(context_lengths, kv_cache, 'o-', label='KV Cache', linewidth=2, markersize=8)
ax.plot(context_lengths, attn_matrix, 's-', label='Attention Matrix (std)', linewidth=2, markersize=8)
ax.plot(context_lengths, activation, '^-', label='Activations', linewidth=2, markersize=8)

ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.set_xlabel('Context Length', fontsize=12)
ax.set_ylabel('Memory (GB)', fontsize=12)
ax.set_title('Memory Scaling by Component', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Add A100 memory line
ax.axhline(y=80, color='red', linestyle='--', linewidth=2, alpha=0.5, label='A100 80GB')

# Standard vs Flash Attention
ax = axes[1]
total_std = [r['total_no_flash_gb'] for r in results]
total_flash = [r['total_with_flash_gb'] for r in results]

ax.plot(context_lengths, total_std, 'o-', label='Standard Attention', 
        linewidth=2, markersize=8, color='#e74c3c')
ax.plot(context_lengths, total_flash, 's-', label='Flash Attention + GQA',
        linewidth=2, markersize=8, color='#2ecc71')

ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.set_xlabel('Context Length', fontsize=12)
ax.set_ylabel('Total Memory (GB)', fontsize=12)
ax.set_title('Total Memory: Standard vs Flash+GQA', fontsize=13, fontweight='bold')
ax.axhline(y=80, color='red', linestyle='--', linewidth=2, alpha=0.5, label='A100 80GB')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nConclusion: Flash Attention + GQA are ESSENTIAL for long contexts")
print("Without them, even 8K context is infeasible on A100")

## 2. RoPE (Rotary Position Embedding)

### Understanding RoPE

**Problem with Absolute Position Embeddings**:
```
Traditional: pos_embed[i] for position i
Issue: Model trained on max_len=2048 cannot handle position 2049!
```

**RoPE Solution** (Su et al., 2021):
```python
# Rotate query and key vectors based on position
def rope(x, position, theta=10000.0):
    # Frequency bands
    freqs = 1.0 / (theta ** (torch.arange(0, d, 2) / d))
    
    # Position-dependent rotation
    angles = position * freqs
    
    # Apply rotation
    cos = torch.cos(angles)
    sin = torch.sin(angles)
    
    # Rotate pairs of dimensions
    x_rotated = rotate_half(x) * sin + x * cos
    return x_rotated
```

**Key Properties**:
- Relative position encoding (attends to relative positions)
- No learned parameters
- Efficient computation
- But: Limited extrapolation beyond training length

In [ ]:
def precompute_freqs_cis(dim: int, end: int, theta: float = 10000.0):
    """
    Precompute RoPE frequencies.
    
    Args:
        dim: Dimension of embeddings
        end: Maximum sequence length
        theta: Base for frequency calculation
    """
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    t = torch.arange(end, device=freqs.device)
    freqs = torch.outer(t, freqs).float()
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)  # complex exponential
    return freqs_cis


def apply_rotary_emb(xq, xk, freqs_cis):
    """
    Apply rotary embeddings to query and key.
    
    Args:
        xq: Query [batch, seq_len, num_heads, head_dim]
        xk: Key [batch, seq_len, num_heads, head_dim]
        freqs_cis: Precomputed frequencies
    """
    # Reshape to complex numbers
    xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))
    
    # Apply rotation
    freqs_cis = freqs_cis[:xq_.shape[1]]
    freqs_cis = freqs_cis.unsqueeze(0).unsqueeze(2)  # [1, seq_len, 1, head_dim/2]
    
    xq_out = torch.view_as_real(xq_ * freqs_cis).flatten(3)
    xk_out = torch.view_as_real(xk_ * freqs_cis).flatten(3)
    
    return xq_out.type_as(xq), xk_out.type_as(xk)


print("RoPE (Rotary Position Embedding)")
print("=" * 70)

# Test RoPE
dim = 128
seq_len = 2048
batch_size = 2
num_heads = 8

# Precompute frequencies
freqs_cis = precompute_freqs_cis(dim, seq_len)

# Create query and key
xq = torch.randn(batch_size, seq_len, num_heads, dim, device=device)
xk = torch.randn(batch_size, seq_len, num_heads, dim, device=device)

# Apply RoPE
xq_rope, xk_rope = apply_rotary_emb(xq, xk, freqs_cis.to(device))

print(f"Input shape: {xq.shape}")
print(f"Output shape: {xq_rope.shape}")
print(f"Frequencies shape: {freqs_cis.shape}")
print("\n✓ RoPE applied successfully")
print("\nBut: RoPE trained on 2K tokens doesn't extrapolate well to 32K!")
print("Solution: RoPE scaling techniques")

## 3. RoPE Scaling Techniques

### Extending Context Beyond Training Length

**Problem**:
```
Model trained with RoPE on 2K tokens
At position 4000: RoPE frequencies unseen during training
Result: Poor attention, degraded quality
```

**Solution 1: Linear Scaling**:
```python
# Scale positions to fit in training range
position_scaled = position / scale_factor

# Example: 8K context, trained on 2K
scale_factor = 8192 / 2048 = 4
position_scaled = position / 4

Pros: Simple, no retraining
Cons: Compresses position information, loses precision
```

**Solution 2: NTK-Aware Scaling**:
```python
# Adjust RoPE base frequency (theta)
theta_new = theta_original * (scale_factor ** (dim / (dim - 2)))

# Changes frequency bands to accommodate longer contexts
Pros: Better preserves position information
Cons: Still approximation
```

**Solution 3: YaRN (Yet another RoPE extensioN)**:
```python
# Separate scaling for different frequency bands
- Low frequencies: Extrapolate (long-range)
- High frequencies: Interpolate (short-range)
- Middle: Gradual transition

Pros: Best quality, handles very long contexts
Cons: More complex, may need fine-tuning
```

In [ ]:
def linear_scaling_rope(dim, end, theta=10000.0, scale_factor=1.0):
    """
    Linear scaling: Compress positions into training range.
    """
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    t = torch.arange(end, device=freqs.device) / scale_factor  # Scale positions
    freqs = torch.outer(t, freqs).float()
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)
    return freqs_cis


def ntk_scaling_rope(dim, end, theta=10000.0, scale_factor=1.0):
    """
    NTK-aware scaling: Adjust base frequency.
    """
    # Adjust theta based on scale factor
    theta_new = theta * (scale_factor ** (dim / (dim - 2)))
    
    freqs = 1.0 / (theta_new ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    t = torch.arange(end, device=freqs.device)
    freqs = torch.outer(t, freqs).float()
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)
    return freqs_cis


def yarn_scaling_rope(dim, end, theta=10000.0, scale_factor=1.0, 
                      extrapolation_factor=1.0, attn_factor=1.0):
    """
    YaRN scaling: Separate handling of frequency bands.
    
    Simplified version for demonstration.
    """
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    
    # Compute wavelengths
    wavelengths = 2 * math.pi / freqs
    
    # Low freq: extrapolate, High freq: interpolate
    low_freq_factor = extrapolation_factor
    high_freq_factor = 1.0 / scale_factor
    
    # Blend factors based on wavelength
    # (This is a simplified version; real YaRN is more sophisticated)
    blend = torch.sigmoid((wavelengths - wavelengths.median()) / wavelengths.std())
    scaling = blend * low_freq_factor + (1 - blend) * high_freq_factor
    
    freqs_scaled = freqs / scaling
    
    t = torch.arange(end, device=freqs.device)
    freqs = torch.outer(t, freqs_scaled).float()
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)
    return freqs_cis


print("RoPE Scaling Methods")
print("=" * 70)

# Parameters
dim = 128
train_len = 2048
target_len = 8192
scale_factor = target_len / train_len

print(f"Training length: {train_len}")
print(f"Target length: {target_len}")
print(f"Scale factor: {scale_factor}x\n")

# Compare different scaling methods
freqs_original = precompute_freqs_cis(dim, train_len)
freqs_linear = linear_scaling_rope(dim, target_len, scale_factor=scale_factor)
freqs_ntk = ntk_scaling_rope(dim, target_len, scale_factor=scale_factor)
freqs_yarn = yarn_scaling_rope(dim, target_len, scale_factor=scale_factor)

print("Method                  Description")
print("-" * 70)
print("Original RoPE:          Frequencies for training length only")
print("Linear Scaling:         Compress positions by 1/scale_factor")
print("NTK-Aware:              Adjust base frequency theta")
print("YaRN:                   Separate scaling for frequency bands")
print("\n✓ All methods generated successfully")

In [ ]:
# Visualize RoPE scaling
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Extract frequency magnitudes at different positions
positions_to_show = [0, 1024, 2048, 4096, 8192]
freq_idx = np.arange(0, dim // 2)

methods = [
    ('Original (2K)', freqs_original, axes[0, 0]),
    ('Linear Scaling', freqs_linear, axes[0, 1]),
    ('NTK-Aware', freqs_ntk, axes[1, 0]),
    ('YaRN', freqs_yarn, axes[1, 1]),
]

for name, freqs, ax in methods:
    for pos in positions_to_show:
        if pos < freqs.shape[0]:
            magnitudes = torch.abs(freqs[pos]).cpu().numpy()
            ax.plot(freq_idx, magnitudes, label=f'Pos {pos}', alpha=0.7)
    
    ax.set_xlabel('Frequency Index', fontsize=11)
    ax.set_ylabel('Magnitude', fontsize=11)
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey Differences:")
print("• Original: Only has frequencies up to training length")
print("• Linear: All positions compressed uniformly")
print("• NTK-Aware: Smooth frequency adjustment")
print("• YaRN: Different scaling for different frequency bands")

## 4. Attention Pattern Analysis

### Visualizing Long Context Attention

In [ ]:
def generate_attention_pattern(seq_len, pattern_type='causal'):
    """
    Generate different attention patterns for visualization.
    """
    if pattern_type == 'causal':
        # Lower triangular (each position attends to past)
        mask = torch.tril(torch.ones(seq_len, seq_len))
    elif pattern_type == 'sliding_window':
        # Local attention window
        window_size = 256
        mask = torch.zeros(seq_len, seq_len)
        for i in range(seq_len):
            start = max(0, i - window_size)
            mask[i, start:i+1] = 1
    elif pattern_type == 'sparse':
        # Sparse attention (e.g., Longformer-style)
        mask = torch.zeros(seq_len, seq_len)
        # Local attention
        for i in range(seq_len):
            start = max(0, i - 32)
            mask[i, start:i+1] = 1
        # Global attention (every 64 tokens)
        mask[:, ::64] = 1
    
    return mask


# Visualize different attention patterns
seq_len = 1024
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

patterns = [
    ('Causal (Standard)', 'causal'),
    ('Sliding Window', 'sliding_window'),
    ('Sparse (Longformer-style)', 'sparse'),
]

for ax, (name, pattern_type) in zip(axes, patterns):
    mask = generate_attention_pattern(seq_len, pattern_type)
    
    # Downsample for visualization
    step = 4
    mask_viz = mask[::step, ::step]
    
    im = ax.imshow(mask_viz, cmap='Blues', aspect='auto', interpolation='nearest')
    ax.set_xlabel('Key Position', fontsize=11)
    ax.set_ylabel('Query Position', fontsize=11)
    ax.set_title(name, fontsize=12, fontweight='bold')
    
    # Show actual positions
    ticks = [0, seq_len // step // 4, seq_len // step // 2, 
             3 * seq_len // step // 4, seq_len // step - 1]
    tick_labels = [0, seq_len // 4, seq_len // 2, 3 * seq_len // 4, seq_len]
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    ax.set_xticklabels(tick_labels)
    ax.set_yticklabels(tick_labels)

plt.tight_layout()
plt.show()

print("\nAttention Pattern Trade-offs:")
print("\nCausal (Standard):")
print("  • Full attention to all previous tokens")
print("  • O(n²) computation and memory")
print("  • Best quality but expensive")
print("\nSliding Window:")
print("  • Only attend to recent tokens")
print("  • O(n × window) computation")
print("  • Good for local patterns (code, text)")
print("\nSparse (Longformer/BigBird):")
print("  • Local + global attention")
print("  • O(n × (local + global)) computation")
print("  • Balance of efficiency and quality")

## 5. Combining All Techniques

### Complete Long Context Solution

In [ ]:
print("Complete Long Context Stack")
print("=" * 80)

stack = """
LAYER 1: Memory Optimization
├─ Flash Attention: O(n²) → O(n) memory for attention
│  Result: 2-4x memory reduction, enables 4-8x longer contexts
│
├─ GQA (Grouped-Query Attention): Reduce KV cache 4-8x
│  Result: num_kv_heads=8 → 8x smaller KV cache
│
└─ Combined: Flash + GQA
   Result: 8-32x total memory reduction!


LAYER 2: Position Encoding
├─ RoPE: Rotary position embeddings (relative positions)
│
├─ RoPE Scaling: Extend beyond training length
│  Options:
│  • Linear Scaling: Simple, works for 2-4x extension
│  • NTK-Aware: Better quality, works for 4-8x
│  • YaRN: Best quality, works for 8-16x
│
└─ Result: Can extend 2K model to 32K context


LAYER 3: Optional Further Optimizations
├─ Sliding Window Attention: Further reduce computation
│  Use case: Very long documents (100K+ tokens)
│
├─ KV Cache Quantization: INT8 or INT4 for KV cache
│  Result: 2-4x additional memory reduction
│
├─ Multi-GPU: Tensor parallelism for very large models
│  Use case: 70B+ models with 32K+ context
│
└─ Prefix Caching: Share common prefixes across requests
   Use case: Multi-turn conversations, batch inference


TYPICAL CONFIGURATION (32K context, Llama 2 70B):
✓ Flash Attention v2
✓ GQA with 8 KV heads
✓ YaRN RoPE scaling (16x extension)
✓ FP16 inference
✓ Optional: KV cache INT8

Result:
• Memory: ~20-30 GB KV cache (vs 88 GB without optimizations)
• Speed: Comparable to short context (Flash Attention)
• Quality: <1% degradation from optimizations
"""

print(stack)

In [ ]:
# Calculate memory with all optimizations
print("\nMemory Comparison: Standard vs Optimized")
print("=" * 80)

seq_len = 32768  # 32K context

# Standard: MHA, no Flash Attention
standard_config = {
    'num_heads': 64,
    'num_kv_heads': 64,  # MHA
    'head_dim': 128,
    'num_layers': 80,
    'd_model': 8192,
}

# Optimized: GQA, Flash Attention
optimized_config = {
    'num_heads': 64,
    'num_kv_heads': 8,  # GQA
    'head_dim': 128,
    'num_layers': 80,
    'd_model': 8192,
}

standard_mem = calculate_long_context_memory(seq_len, standard_config)
optimized_mem = calculate_long_context_memory(seq_len, optimized_config)

print(f"{'Configuration':<20} {'KV Cache (GB)':<15} {'Total w/o Flash':<18} {'Total w/ Flash'}")
print("-" * 80)
print(f"{'Standard (MHA)':<20} {standard_mem['kv_cache_gb']:<15.1f} "
      f"{standard_mem['total_no_flash_gb']:<18.1f} {standard_mem['total_with_flash_gb']:.1f}")
print(f"{'Optimized (GQA+Flash)':<20} {optimized_mem['kv_cache_gb']:<15.1f} "
      f"{optimized_mem['total_no_flash_gb']:<18.1f} {optimized_mem['total_with_flash_gb']:.1f}")

kv_reduction = standard_mem['kv_cache_gb'] / optimized_mem['kv_cache_gb']
total_reduction = standard_mem['total_with_flash_gb'] / optimized_mem['total_with_flash_gb']

print(f"\nReductions:")
print(f"  KV Cache: {kv_reduction:.1f}x smaller")
print(f"  Total Memory: {total_reduction:.1f}x smaller")
print(f"\nFeasibility on A100 80GB:")
print(f"  Standard: ✗ {standard_mem['total_with_flash_gb']:.0f} GB (OOM)")
print(f"  Optimized: ✓ {optimized_mem['total_with_flash_gb']:.0f} GB (fits with model!)")

## 6. Practical Example: Long Document Analysis

In [ ]:
print("Long Document Analysis - Practical Guide")
print("=" * 80)

guide = """
SCENARIO: Analyze a 30-page research paper (~15K tokens)

SETUP:

from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model with long context support
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-13b-hf",
    torch_dtype=torch.float16,
    device_map="auto",
    attn_implementation="flash_attention_2",  # Flash Attention
    rope_scaling={  # YaRN scaling
        "type": "yarn",
        "factor": 8.0,  # 4K → 32K
    },
)

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-13b-hf")


USAGE:

# Long document
document = """[30-page research paper text...]"""

# Tokenize (will be ~15K tokens)
inputs = tokenizer(document, return_tensors="pt", truncation=True, max_length=32768)
inputs = inputs.to("cuda")

# Generate summary
prompt = f"{document}\n\nProvide a comprehensive summary of this paper:"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=500,
    do_sample=True,
    temperature=0.7,
    use_cache=True,  # Use KV cache
)

summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(summary)


MEMORY MONITORING:

import torch

# Before generation
torch.cuda.reset_peak_memory_stats()
start_mem = torch.cuda.memory_allocated() / 1e9

# Generate
outputs = model.generate(**inputs, max_new_tokens=500)

# After generation
end_mem = torch.cuda.memory_allocated() / 1e9
peak_mem = torch.cuda.max_memory_allocated() / 1e9

print(f"Start memory: {start_mem:.2f} GB")
print(f"End memory: {end_mem:.2f} GB")
print(f"Peak memory: {peak_mem:.2f} GB")
print(f"KV cache size: {end_mem - start_mem:.2f} GB")


PERFORMANCE TIPS:

1. Use Flash Attention (2-4x faster)
2. Use GQA model if available (8x less KV cache)
3. Enable RoPE scaling for contexts beyond training length
4. Use FP16 (2x less memory vs FP32)
5. Consider batching multiple documents for throughput
6. Monitor memory usage to avoid OOM
7. For 100K+ tokens: Use vLLM with PagedAttention


EXPECTED PERFORMANCE (Llama 2 13B, 15K context, A100):
• Memory: ~8-12 GB total (model + KV cache)
• Speed: ~50-100 tokens/sec
• Quality: Comparable to short context
"""

print(guide)

## 7. Benchmarking and Best Practices

In [ ]:
print("Long Context Best Practices")
print("=" * 80)

best_practices = """
MEMORY OPTIMIZATION:

1. Always use Flash Attention for contexts > 2K
   • 2-4x memory reduction
   • 1.5-2x speedup
   • Required: CUDA 11.6+, Ampere+ GPU

2. Prefer GQA models over MHA for long contexts
   • 4-8x smaller KV cache
   • Minimal quality loss (<1%)
   • Available: Llama 2 70B, Mistral 7B

3. Use appropriate RoPE scaling
   • 2-4x extension: Linear scaling
   • 4-8x extension: NTK-aware
   • 8-16x extension: YaRN
   • >16x: May need fine-tuning

4. Consider KV cache quantization for extreme lengths
   • INT8: 2x reduction, <0.5% quality loss
   • INT4: 4x reduction, 1-2% quality loss


CONTEXT LENGTH GUIDE:

Target Context  | Required Techniques                | Hardware
----------------|-----------------------------------|------------------
2K - 4K         | Flash Attention                   | RTX 3090, A10
4K - 8K         | Flash + GQA                       | A10, A100 40GB
8K - 16K        | Flash + GQA + RoPE scaling        | A100 40GB
16K - 32K       | All above + FP16                  | A100 80GB
32K - 64K       | All above + KV quant (optional)   | A100 80GB, H100
64K - 128K      | All above + Multi-GPU             | 2x A100, H100
128K+           | Specialized (sparse attn, etc.)   | Multi-GPU


QUALITY VALIDATION:

Test on benchmarks with different context lengths:
1. Needle in haystack: Can model find specific info in long context?
2. Long document QA: Accuracy on questions requiring full context
3. Summarization: Quality of summaries for long documents
4. Multi-turn dialogue: Coherence over many turns

Expected quality retention:
• Flash Attention: No loss (exact computation)
• GQA-8: <1% loss
• RoPE scaling (YaRN): <1% loss
• KV cache INT8: <0.5% loss
• Total: <2% loss (acceptable for most use cases)


PRODUCTION DEPLOYMENT:

1. Use vLLM for serving
   • Automatic Flash Attention and GQA
   • PagedAttention for memory efficiency
   • Continuous batching for throughput

2. Monitor memory usage
   • Set max_model_len based on available memory
   • Leave headroom for batch processing
   • Use memory profiling tools

3. Load balancing
   • Short contexts: Higher batch size
   • Long contexts: Lower batch size
   • Dynamic batching based on context length

4. Caching strategies
   • Prefix caching for common prompts
   • KV cache reuse across similar requests
   • Document chunking for very long inputs
"""

print(best_practices)

In [ ]:
# Create decision tree visualization
fig, ax = plt.subplots(figsize=(12, 8))
ax.axis('off')

# Decision tree text
decision_tree = """
                        Target Context Length?
                                |
        ________________|________________
       |                |                |
    2K-8K            8K-32K           32K+
       |                |                |
  Flash Attn      Flash + GQA      Flash + GQA
       |           + RoPE Scaling   + YaRN
       |                |           + Multi-GPU?
       |                |                |
   ✓ Good          ✓ Good           ✓ Good
   RTX 3090       A100 40GB         A100 80GB/H100


Quick Checks:
• Context > 2K? → Use Flash Attention (always)
• Context > 8K? → Use GQA model (8x KV reduction)
• Context > training length? → Use RoPE scaling
• Still OOM? → Quantize KV cache or use multi-GPU
"""

ax.text(0.5, 0.5, decision_tree, fontsize=11, family='monospace',
        ha='center', va='center', wrap=True)
ax.set_title('Long Context Decision Tree', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

## Summary

### Key Takeaways

1. **The Challenge**:
   - Memory scales linearly with context length
   - 32K context on Llama 2 70B: 88 GB KV cache (standard MHA)
   - Exceeds single GPU capacity without optimization

2. **Memory Optimization Stack**:
   - **Flash Attention**: 2-4x memory reduction (O(n²) → O(n))
   - **GQA**: 4-8x KV cache reduction (8 KV heads vs 64)
   - **Combined**: 8-32x total memory savings
   - Result: 32K context fits on A100 80GB

3. **RoPE Scaling**:
   - Problem: Models trained on 2K-4K cannot extrapolate to 32K
   - **Linear Scaling**: Simple, works for 2-4x extension
   - **NTK-Aware**: Better quality, works for 4-8x
   - **YaRN**: Best quality, works for 8-16x extension

4. **Performance** (Llama 2 70B, 32K context, A100):
   - Memory: ~25-30 GB KV cache (with Flash + GQA)
   - Speed: 50-100 tokens/sec
   - Quality: <2% degradation from all optimizations

5. **Practical Guidelines**:
   - 2K-8K: Flash Attention only
   - 8K-32K: Flash + GQA + RoPE scaling
   - 32K+: Add KV quantization or multi-GPU
   - Always use Flash Attention for contexts > 2K

6. **Real-World Examples**:
   - Document analysis: 15-30K tokens (research papers, legal docs)
   - Code repositories: 20-50K tokens (entire codebases)
   - Multi-turn conversations: 10-30K tokens (chat history)
   - Book summarization: 50-100K tokens

### Historical Timeline

- **2021**: RoPE introduced, models limited to 2-4K
- **2022**: Flash Attention enables 8-16K contexts
- **2023**: GQA + RoPE scaling → 32K+ contexts
- **2023**: Claude 2 (100K), GPT-4 (32K)
- **2024**: Gemini 1.5 (1M tokens!) - specialized architecture

### Implementation Checklist

✓ **Always**:
- Use Flash Attention (flash_attention_2 or F.scaled_dot_product_attention)
- Choose GQA models for long contexts (Llama 2 70B, Mistral)
- Monitor memory usage
- Use FP16 inference

✓ **For Contexts > Training Length**:
- Configure RoPE scaling (YaRN for best quality)
- Validate quality on long-context benchmarks
- Test needle-in-haystack tasks

✓ **For Extreme Lengths (64K+)**:
- Consider KV cache quantization
- Use multi-GPU if needed
- Profile memory carefully
- May need sparse attention patterns

### Next Steps

- **Notebook 20**: Flash Attention (attention computation optimization)
- **Notebook 21**: MQA/GQA (KV cache reduction)
- **Notebook 24**: Memory Optimization Comparison (all techniques)
- **Notebook 55**: KV Cache Optimization (quantization, compression)

### Further Reading

- Paper: "Extending Context Window of Large Language Models via Position Interpolation" (Chen et al., 2023)
- Paper: "YaRN: Efficient Context Window Extension" (Peng et al., 2023)
- Paper: "Longformer: The Long-Document Transformer" (Beltagy et al., 2020)
- LLM_INFERENCE_ROADMAP.md - Stage 2: Memory Optimization